# Schema design for reliability

**Scenario:** a live match service turns caster commentary into per-map player rows. A dashboard adds
up kills and groups players by role. On finals night it shows a player in a role that does not exist,
and the kill total refuses to add up at all.

The shape was forced. Every row was valid JSON. That was never the part that was going to break.

A schema is a written promise about a value, and each keyword promises something different. Think of
it as a drop-down list instead of a text box. A text box takes anything anyone types.

## Mechanics

Five keywords, and what each one is worth.

| Keyword | What it promises | What it does not promise |
|---|---|---|
| `type` | the value is a string, a number, a boolean | that the value is right |
| `enum` | the value is one of a listed set | that the model picked the correct one |
| `required` | the key is present in the object | that the value behind it is known |
| `additionalProperties: false` | no keys you never asked for | anything about the keys you did |
| `["integer", "null"]` | the key is present and may be an honest blank | |

The two that decide whether a pipeline holds are `enum` and `required`. Without `enum`, a field a
report groups by is free text. Without `required`, a field a report needs can simply be absent.

## The picture

![A loose schema and a tight one, from the same transcript](images/schema-tightness.svg)

Same model, same commentary, two schemas. Only one of them produces rows a dashboard can add.

## The cost

```
cost = rows the report cannot add + rows it adds to a group that does not exist
```

The second half is worse than the first. A row that will not add crashes something and gets noticed.
A row filed under an invented role is counted, charted and believed.

## The failure

Three pieces of commentary from one series, including one about a player who did not play.

In [1]:
CASTS = [
    "Okay so Nyx on Jett went absolutely nuclear that round, double-digit frags, team took the map.",
    "Vex was IGLing from the back, barely touched anyone, maybe two or three, they threw it at the end.",
    "Riko subbed in as coach for map three, no stats to speak of, series ended level.",
]

The schema below is what a first pass usually looks like. Every field is there, every type is a
string, and nothing is marked required.

In [2]:
LOOSE = {"type": "function", "function": {
    "name": "log_performance",
    "description": "Log one player's map performance.",
    "parameters": {"type": "object", "properties": {
        "player": {"type": "string"},
        "role": {"type": "string"},
        "kills": {"type": "string"},
        "outcome": {"type": "string"}}}}}

One call per piece of commentary, forced through the tool so the shape is never in doubt.

In [3]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("08-deterministic-outputs/02-schema-design-for-reliability")


def log_row(cast, tool):
    """Force one row out of one piece of commentary."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300, tools=[tool],
        tool_choice={"type": "function", "function": {"name": "log_performance"}},
        messages=[{"role": "system", "content": "You log esports match telemetry."},
                  {"role": "user", "content": cast}])
    return json.loads(reply.choices[0].message.tool_calls[0].function.arguments)

Now the thing the dashboard does. Add the kills, and group by role.

In [4]:
loose_rows = [log_row(cast, LOOSE) for cast in CASTS]
for row in loose_rows:
    print(f"  {row}")

kills = [row["kills"] for row in loose_rows]
print(f"\ndistinct roles: {sorted({row['role'] for row in loose_rows})}")
print(f"kills as returned: {kills}")
assert all(k.isdigit() for k in kills), f"the dashboard cannot add {kills}"

  {'role': 'Jett', 'outcome': 'won', 'player': 'Nyx', 'kills': '10+'}
  {'outcome': 'lost', 'role': 'IGL', 'kills': '2-3', 'player': 'Vex'}
  {'kills': '0', 'outcome': 'loss', 'player': 'Riko', 'role': 'coach'}

distinct roles: ['IGL', 'Jett', 'coach']
kills as returned: ['10+', '2-3', '0']


AssertionError: the dashboard cannot add ['10+', '2-3', '0']

## The diagnosis

Every row is valid against the schema that was sent, and the report cannot use a single one.

**`kills` was typed as a string**, so `10+` and `2-3` are legal. They are honest readings of
"double-digit frags" and "maybe two or three", which is why the model wrote them. Nothing can add
them.

**`role` had no `enum`**, so the model returned whatever the commentary called the player. One is an
agent name and one is a job that is not a playing role. `outcome` went the same way, and a field with
two possible values came back in three spellings.

**Nothing was `required`**, so any of these keys could have been missing instead, and the same code
would have raised a `KeyError` on a different night.

None of that is the model being wrong. The schema described a shape and promised nothing about
content, so the model filled every field with the most reasonable text it could find.

## The fix

The same four fields, with the promises turned up. Closed sets where a set exists, a number where a
report will do arithmetic, and every key required.

In [5]:
TIGHT = {"type": "function", "function": {
    "name": "log_performance",
    "description": "Log one player's map performance.",
    "parameters": {"type": "object", "properties": {
        "player": {"type": "string"},
        "role": {"type": "string", "enum": ["duelist", "controller", "initiator", "sentinel"]},
        "kills": {"type": "integer", "minimum": 0},
        "outcome": {"type": "string", "enum": ["win", "loss", "draw"]}},
        "required": ["player", "role", "kills", "outcome"],
        "additionalProperties": False}}}

Same commentary, same model, same forced call. Only the schema changed.

In [6]:
tight_rows = [log_row(cast, TIGHT) for cast in CASTS]
for row in tight_rows:
    print(f"  {row}")

print(f"\nbefore: 0 of {len(CASTS)} rows could be added, roles were free text")
print(f"after : {len(tight_rows)} of {len(CASTS)} rows added to "
      f"{sum(row['kills'] for row in tight_rows)} kills")

  {'kills': 10, 'outcome': 'win', 'role': 'duelist', 'player': 'Nyx'}
  {'kills': 3, 'role': 'controller', 'outcome': 'loss', 'player': 'Vex'}
  {'role': 'sentinel', 'kills': 0, 'player': 'Riko', 'outcome': 'draw'}

before: 0 of 3 rows could be added, roles were free text
after : 3 of 3 rows added to 13 kills


Read the third row again. Riko was a coach and never played, and the schema now insists on a role
from the list, so the model picked one. The report gained a player who was not on the server.

An `enum` removes every wrong-looking answer and leaves the model no way to say it does not know.
Make not knowing a legal value instead, with a nullable type and a description saying when to use
it.

In [7]:
import copy

HONEST = copy.deepcopy(TIGHT)
fields = HONEST["function"]["parameters"]["properties"]
fields["role"]["type"] = ["string", "null"]
fields["role"]["enum"] = fields["role"]["enum"] + [None]
fields["role"]["description"] = "null when the commentary states no playing role"
fields["kills"]["type"] = ["integer", "null"]
fields["kills"]["description"] = "null when no kill count is stated"

The key is still required, so a row never arrives with the field missing. What changed is that the
model can now put a blank in it and mean it.

In [8]:
honest_rows = [log_row(cast, HONEST) for cast in CASTS]
for row in honest_rows:
    print(f"  {row}")

played = [row for row in honest_rows if row["role"] is not None]
print(f"\ntight schema  : {len(tight_rows)} rows, {len(tight_rows)} of them claim a role")
print(f"honest schema : {len(honest_rows)} rows, {len(played)} of them claim a role")

  {'outcome': 'win', 'role': 'duelist', 'kills': 10, 'player': 'Nyx'}
  {'outcome': 'loss', 'role': 'sentinel', 'player': 'Vex', 'kills': 3}
  {'player': 'Riko', 'role': None, 'outcome': 'draw', 'kills': None}

tight schema  : 3 rows, 3 of them claim a role
honest schema : 3 rows, 2 of them claim a role


## The gate

The regression is a new field arriving as free text because nobody remembered the rule. This test
walks the schema itself, so it catches a field that was never added to `required` and a closed set
that was shipped without an `enum`. No model, no key, no network.

In [9]:
CLOSED_SETS = ("role", "outcome")


def test_every_field_is_pinned_down():
    params = HONEST["function"]["parameters"]
    assert params["additionalProperties"] is False, "unlisted keys can arrive"
    assert set(params["required"]) == set(params["properties"]), "a field can go missing"
    for name in CLOSED_SETS:
        assert "enum" in params["properties"][name], f"{name} is free text again"


test_every_field_is_pinned_down()
print("gate holds: every key required, every closed set listed, nothing else allowed")

gate holds: every key required, every closed set listed, nothing else allowed


Add a field to `properties` and leave it out of `required`, and this test fails on the second line.

What it cannot catch is a value that passes every rule and is still wrong. A row reading twelve kills
for a player who scored three satisfies this schema completely. Code that rejects a value that is the
right shape but the wrong answer is a validator, and that is the next vault.

### Enterprise exploration

- The roles list changes when the publisher adds an agent class. Where does it live so the schema,
  the dashboard and the migration cannot disagree?
- A null role is honest and a report still has to show something. Who decides whether an unknown row
  is dropped or counted?
- Finals night is a hundred times a normal night. Does a tighter schema cost more per row, and what
  would you measure rather than guess?
- Player data feeds team contracts and betting markets. What is the compliance exposure of an
  invented role reaching a public dashboard?

### Key takeaways

- `type` promises a shape. It never promises the value is right.
- Without `enum`, any field a report groups by is free text.
- Without `required`, a field a report needs can simply be absent.
- A closed set with no blank in it forces the model to guess. Make not knowing a legal answer.